In [1]:
import pandas as pd

df = pd.read_csv('rfm_data.csv')

df.head()

,order_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,payment_type,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,credit_card,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,voucher,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,voucher,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,boleto,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,credit_card,179.12


In [2]:
# 1. Convert the timestamp column to Python's datetime format
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Define the 'current date' for Recency calculation (1 day after the latest purchase)
recent_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# 3. Group by customer to calculate Recency, Frequency, and Monetary (RFM) values
rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (recent_date - x.max()).days, # Recency: Days since last purchase
    'order_id': 'nunique',                                              # Frequency: Total number of unique orders
    'payment_value': 'sum'                                              # Monetary: Total amount spent
}).reset_index()

# 4. Rename columns for clarity
rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

# Display the first 5 rows of the new RFM dataframe
rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [3]:
# 1. Assign R and M scores (1-5) using quantiles (qcut divides data into 5 equal parts)
# Note: Lower Recency is better, so labels are reversed [5, 4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# 2. Assign F score. (Using rank to avoid duplicate bin errors since most Olist customers buy only once)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

# 3. Combine individual scores into a single RFM Segment string (e.g., '555' for best customers)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Display the assigned scores
rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_Score,M_Score,F_Score,RFM_Segment
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,4,1,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,2,1,112
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,4,1,214


In [4]:
# 1. Define a function to segment customers based on R (Recency) and M (Monetary) scores
# (Olist customers mostly buy once, so F score is less defining for segments here)
def get_segment(row):
    r = row['R_Score']
    m = row['M_Score']

    if r >= 4 and m >= 4:
        return 'Champions (VIP)'
    elif r >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and m <= 2:
        return 'New / Promising'
    elif r <= 2 and m >= 4:
        return 'At Risk (High Value)'
    elif r <= 2 and m <= 2:
        return 'Lost / Low Value'
    else:
        return 'Regular / Average'

# 2. Apply the segmentation rule to create a new column
rfm['Customer_Segment'] = rfm.apply(get_segment, axis=1)

# 3. Calculate how many customers are in each segment to see the big picture
segment_counts = rfm['Customer_Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer_Count']

# Display the final summary table
segment_counts

,Segment,Customer_Count
0,Loyal Customers,18729
1,Champions (VIP),15436
2,Lost / Low Value,15379
3,Regular / Average,14905
4,At Risk (High Value),14488
5,New / Promising,14420
